In [ ]:
import sys
!{sys.executable} -m pip install pymysql pandas -q

In [ ]:
import pymysql
import pandas as pd
from datetime import datetime

# ── DB 연결 정보 입력 ──────────────────────────────────────
DB_HOST     = 'mp.smhrd.or.kr'       # 호스트
DB_PORT     = 3306              # 포트
DB_NAME     = 'lang9_core_overfit'     # DB명
DB_USER     = 'lang9_core_overfit'     # 계정
DB_PASSWORD = 'overfit9' # 비밀번호

# ── CSV 파일 경로 ──────────────────────────────────────────
CSV_PATH = 'cleaned_reviews.csv'   # 전체 리뷰 (ad_flag 포함)

# ── 데이터 로드 ────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f'✅ CSV 로드 완료: {len(df)}개')
print(df.columns.tolist())
print(df.head(3))

In [ ]:
import pymysql
import pandas as pd

# ── DB 연결 ──────────────────────────────────────────────
conn = pymysql.connect(
    host=DB_HOST, port=DB_PORT,
    db=DB_NAME, user=DB_USER,
    password=DB_PASSWORD,
    charset='utf8mb4'
)
cursor = conn.cursor()
print('✅ DB 연결 완료')

# ── 데이터 로드 ──────────────────────────────────────────
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df['브랜드']   = df['브랜드'].fillna('')
df['카테고리'] = df['카테고리'].fillna('')
df['별점']     = pd.to_numeric(df['별점'], errors='coerce').fillna(0)
df['ad_flag']  = df['ad_flag'].fillna('N')
df['리뷰']     = df['리뷰'].fillna('')
print(f'✅ CSV 로드 완료: {len(df)}개')

# ── 제품 중복 제거 ───────────────────────────────────────
products = df[['제품명', '브랜드', '카테고리', '이미지URL']].drop_duplicates(
    subset=['제품명', '브랜드']
).reset_index(drop=True)
print(f'고유 제품 수: {len(products)}개')

# ── t_product 적재 ───────────────────────────────────────
print('\n▶ t_product 적재 중...')
prod_idx_map = {}

product_sql = """
INSERT INTO t_product (prod_name, brand_name, category, img_url, target_skin)
VALUES (%s, %s, %s, %s, %s)
"""

new_prod = 0
skip_prod = 0

for _, row in products.iterrows():
    # 중복 체크
    cursor.execute("""
        SELECT prod_idx FROM t_product
        WHERE prod_name = %s AND brand_name = %s
    """, (str(row['제품명'])[:100], str(row['브랜드'])[:100]))

    existing = cursor.fetchone()

    if existing:
        prod_idx = existing[0]
        skip_prod += 1
    else:
        cursor.execute(product_sql, (
            str(row['제품명'])[:100],
            str(row['브랜드'])[:100],
            str(row['카테고리'])[:50],
            str(row.get('이미지URL', ''))[:255],
            '',
        ))
        prod_idx = cursor.lastrowid
        new_prod += 1

    key = f"{row['제품명']}_{row['브랜드']}"
    prod_idx_map[key] = prod_idx

conn.commit()
print(f'  ✅ t_product 완료 - 신규: {new_prod}개 / 중복 스킵: {skip_prod}개')

# ── t_review 적재 ────────────────────────────────────────
print('\n▶ t_review 적재 중...')

review_sql = """
INSERT INTO t_review (prod_idx, review_content, rating, ad_flag, review_url)
VALUES (%s, %s, %s, %s, %s)
"""

batch = []
new_review  = 0
skip_review = 0

for _, row in df.iterrows():
    key = f"{row['제품명']}_{row['브랜드']}"
    prod_idx = prod_idx_map.get(key)
    if not prod_idx:
        continue

    review_content = str(row['리뷰'])[:5000]

    # 중복 체크 (같은 prod_idx + 같은 리뷰 내용)
    cursor.execute("""
        SELECT review_idx FROM t_review
        WHERE prod_idx = %s AND review_content = %s
    """, (prod_idx, review_content))

    if cursor.fetchone():
        skip_review += 1
        continue

    review_url = str(row.get('리뷰URL', ''))[:255] if '리뷰URL' in df.columns else ''

    batch.append((
        prod_idx,
        review_content,
        float(row['별점']) if row['별점'] else 0.0,
        str(row['ad_flag']),
        review_url,
    ))
    new_review += 1

    # 1000개씩 배치 insert
    if len(batch) >= 1000:
        cursor.executemany(review_sql, batch)
        conn.commit()
        print(f'  {new_review}개 적재됨...')
        batch = []

# 나머지
if batch:
    cursor.executemany(review_sql, batch)
    conn.commit()

print(f'  ✅ t_review 완료 - 신규: {new_review}개 / 중복 스킵: {skip_review}개')

# ── 검증 ─────────────────────────────────────────────────
cursor.execute('SELECT COUNT(*) FROM t_product')
p_count = cursor.fetchone()[0]
cursor.execute('SELECT COUNT(*) FROM t_review')
r_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM t_review WHERE ad_flag='Y'")
ad_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM t_review WHERE ad_flag='N'")
normal_count = cursor.fetchone()[0]

print(f'\n─── DB 적재 결과 ───')
print(f'  t_product : {p_count}개')
print(f'  t_review  : {r_count}개')
print(f'  광고성(Y) : {ad_count}개')
print(f'  일반(N)   : {normal_count}개')

cursor.close()
conn.close()
print('\n✅ DB 연결 종료')